In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import (
    StandardScaler,
    Normalizer,
    FunctionTransformer,
    PowerTransformer,
    OneHotEncoder,
    OrdinalEncoder,
    SplineTransformer,
)
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.feature_extraction import FeatureHasher
from sklearn.compose import make_column_transformer
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import train_test_split, cross_val_score


In [ ]:
# Revisit the OKCupid data
df = pd.read_csv("../04_categorical/profiles_revised.csv")

In [ ]:
# Target: predict if job is stem related
df["job"].value_counts()
df["stem_job"] = df["job"].str.contains("computer|science", case=False, na=False)
df["stem_job"].value_counts() / len(df)

In [ ]:
df.info()

## Understanding missing values
In addition to looking at how many of which feature are missing, it's useful to understand whether missing values tend to co-occur.

In [ ]:
# Store a copy before we mess with it
og_df = df.copy()

In [ ]:
# set income < 0 to missing
# df.loc[df["income"] < 0, "income"] = np.nan
# df.head()

In [ ]:
print("Missing values by percentage of dataset")
df.isna().sum().sort_values(ascending=False) / len(df) * 100

In [ ]:
# How many missing values per sample?
missing_per_sample = df.isna().sum(axis=1).value_counts()
plt.bar(missing_per_sample.index, missing_per_sample)
plt.xlabel("Number of missing features")
plt.ylabel("Number of samples")

In [ ]:
# Maybe a heatmap can show trends?
missing_samples = df.isna().sum(axis=1).sort_values()
missing_samples.drop(missing_samples[missing_samples == 0].index, inplace=True)
missing_features = df.isna().sum(axis=0).sort_values()
missing_features = missing_features[missing_features > 0]
sns.heatmap(df.loc[missing_samples.index, missing_features.index].isna())

In [ ]:
# how about correlations between missing values?
missing_cor = df.loc[missing_samples.index, missing_features.index].isna().corr()
# zero out the diagonal so we can visualize it better
missing_cor -= np.identity(missing_cor.shape[0])
sns.heatmap(missing_cor)

In [ ]:
# before looking at relationship to target, split the data
train, test = train_test_split(df, stratify=df["stem_job"], random_state=12345)

In [ ]:
# look at the ones that have a lot missing
train_missing = train.isna().sum(axis=0).sort_values(ascending=False)
# filter down to the > 20% missing
n_missing = train_missing[train_missing > 0.2 * len(train)]
print(n_missing)

# percentage in each target category
train_prop = train["stem_job"].value_counts() / len(train)
fig, axes = plt.subplots(2, 3, figsize=(12, 6), sharex=True, sharey=True)
for feat, ax in zip(n_missing.index, axes.flatten()):
    counts = train.groupby("stem_job")[feat].apply(lambda x: x.isna().sum())
    # normalize
    counts /= n_missing[feat]
    # compare to the overall proportion of stem/not stem
    ax.bar([-0.1, .9], train_prop, width=0.2, label="Overall")
    ax.bar([0.1, 1.1], counts, width=0.2, label=f"{feat}")
    ax.set_xticks([0, 1], labels=["Non-STEM", "STEM"])
    ax.set_title(f"Proportion of missing {feat}")
    ax.legend()

# Look at value_counts for the features with lots of missing data, grouped by stem_job


## Dealing with missing values

In [ ]:
# select features to use in the model
numeric_features = ["age", "height", "income"]
cat_features = ["drinks", "education", "sex"]

# df.dropna(subset=numeric_features + cat_features, inplace=True)

X = df[numeric_features + cat_features].copy()
y = df["stem_job"].copy()

# split!
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=df["stem_job"], random_state=12345
)

# print percentage in each class to double check stratification
print(y_train.value_counts() / len(y_train))
print(y_test.value_counts() / len(y_test))

In [ ]:
# Define the trickier encoders
drink_enc = OrdinalEncoder(
            categories=[
                [
                    "not at all",
                    "rarely",
                    "socially",
                    "often",
                    "very often",
                    "desperately",
                ]
            ],
            handle_unknown="use_encoded_value",
            unknown_value=-1,
        )

# This took a while mucking around to figure out the right magic between df/series
def split_edu(df):
    df["education"] = df["education"].str.split(" ")
    return df["education"]

edu_enc = make_pipeline(
        FunctionTransformer(split_edu, validate=False),
        FeatureHasher(n_features=8, input_type="string"),
    )


In [ ]:
df["education"].str.split(" ")

In [ ]:
# Build the preprocessing pipeline

numeric_pipeline = make_pipeline(
    SimpleImputer(strategy="mean", add_indicator=True),
    PowerTransformer(method="yeo-johnson"),
)

preprocessor = make_column_transformer(
    (PowerTransformer(method="yeo-johnson"), numeric_features),
    (OneHotEncoder(handle_unknown="ignore"), ["sex"]), # sparse_output = False for pandas output
    (drink_enc, ["drinks"]),
    (edu_enc, ["education"]), # can't be converted to pandas
    
)#.set_output(transform="pandas")
preprocessor

In [ ]:
type(X_train)

In [ ]:
preprocessor.fit_transform(X_train, y_train)

In [ ]:
# Now add on a model!
pipeline = make_pipeline(
    preprocessor,
    SGDClassifier(class_weight="balanced"), #class_weight="balanced" does some magic
)

cross_val_score(pipeline, X_train, y_train, scoring="recall")

## Extra plots used in slides

In [ ]:
# assume -1 is encoding missingness
plt.hist(df[df["income"] > 0]["income"], bins=50)
plt.xlabel("Income > 0 in OKCupid Dataset")
plt.ylabel("Number of instances")
plt.savefig("../../static/img/06-income-hist.png")

In [ ]:
# What about boxplots vs STEM?
# Only use the training data since we're looking at relationship with predictor
subset = X_train["income"] > 0
plot_df = pd.DataFrame({
    "income": X_train.loc[subset, "income"],
    "stem": y_train.loc[subset].astype(bool)
})
fig, ax = plt.subplots()
plot_df.plot.box(ax=ax,by="stem")
ax.set_xlabel("STEM job")
ax.set_xticks([1,2], labels=["Non-STEM", "STEM"])
ax.set_ylabel("Income (> 0)")
plt.yscale("log")
plt.savefig("../../static/img/06-income-boxplot.png")

In [ ]:
# Those outliers are all overlapping, try spreading them out with jitter
fig, ax = plt.subplots()
plot_df.query("income < 140000").plot.box(ax=ax,by="stem")
ax.set_xlabel("STEM job")
ax.set_xticks([1,2], labels=["Non-STEM", "STEM"])
ax.set_ylabel("Income (> 0)")
plt.yscale("log")

# we need a random number generator
rng = np.random.default_rng(seed=67)
outliers = plot_df.query("income >= 140000")
jit_x = rng.normal(size=outliers.shape[0], scale=0.1) + 1 + outliers["stem"] * 1
jit_y = rng.normal(size=outliers.shape[0], scale=10000) + outliers["income"] 
plt.scatter(jit_x, jit_y, alpha=0.3)
